# event_systemwide

This notebook extracts, processes, and classifies HMIS event-level data—including enrollments, prior living situations, 
exit destinations, current living situations, and service records into a unified event stream. This table supports tracking
client flow in and out of the homelessness system.

**OUTLINE**
1. **Import Dependencies**: Load required packages for data processing, API access, and Azure connections.
2. **Connect to Looker API**: Initialize Looker SDK to pull HMIS datasets.
3. **Define Classification Lists**: Create reference lists for categorizing living situations, destinations, and project types.
4. **Extract data from Looker API**:
    - Enrollments
    - Current Living Situation (CLS)
    - Services
4. **Create Event Data**: 
    - CLS events
    - Homeless events from 
        - Enrollments 
        - Prior living situations
        - Currently fleeing
    - Housed events from move-in dates from PH programs
    - Housed events from homeless prevention enrollment start dates
    - Service events
    - Other Enrollment Events
        - Open enrollments
        - Exits 
    - Deceased Events
5. **Compile Events**
6. **Output Results**: Store final table in Azure Blob Storage for piping into SQL sever

> **Note**: Make sure you have access to Looker API credentials and connected Azure resources before running.


In [ ]:
import pandas as pd
import time 
import numpy as np
import looker_sdk as sdk
from looker_sdk import api_settings
from looker_sdk import models40, init40
from looker_sdk import init40
import looker_sdk
from io import StringIO
import re
from datetime import timedelta
from datetime import date
from io import BytesIO


import io
import os
from azure.storage.blob import BlobServiceClient
from azure.identity import DefaultAzureCredential


## Connect to Looker API

In [ ]:
# #Add chunk to connect to Looker API in production

# class [[ORG_PREFIX]]_Looker_API_Settings(api_settings.ApiSettings):
#     def __init__(self, *args, **kw_args):
#         self.my_var = kw_args.pop("my_var")
#         super().__init__(*args, **kw_args)

#     def read_config(self) -> api_settings.SettingsConfig:
#         config = super().read_config()
#         # See api_settings.SettingsConfig for required fields.
#         if self.my_var == "set":
#             config["base_url"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','[[LOOKER_BASE_URL_SECRET]]')
#             config["client_id"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','[[LOOKER_CLIENT_ID_SECRET]]')
#             config["client_secret"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','[[LOOKER_CLIENT_SECRET_SECRET]]')          
#         return config

# sdk = looker_sdk.init40(config_settings=[[ORG_PREFIX]]_Looker_API_Settings(my_var="set"))

# vars(vars(vars(sdk)["transport"])["settings"])["timeout"] = 300

In [ ]:
#Comment out in production. Use for development

# Import Looker SDK connection from shared script
import sys
sys.path.insert(0, '..')
from [[LOCAL_SCRIPT_NAME]] import sdk

In [ ]:
# Helper function to retry Looker queries on timeout with exponential backoff
def run_query_with_retry(sdk, body, result_format="csv", max_retries=5, base_delay=15):
    """
    Run a Looker inline query with retry logic for timeouts.
    
    Args:
        sdk: Looker SDK instance
        body: Query body dictionary
        result_format: Output format (default: "csv")
        max_retries: Maximum number of retry attempts (default: 5)
        base_delay: Base delay in seconds between retries (default: 15)
    
    Returns:
        Query result
    
    Raises:
        Exception: If all retries are exhausted
    """
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return sdk.run_inline_query(result_format=result_format, body=body)
        except Exception as e:
            last_error = e
            error_str = str(e).lower()
            # Check if it's a timeout or server error worth retrying
            if "timeout" in error_str or "504" in str(e) or "503" in str(e) or "502" in str(e) or "timed out" in error_str:
                if attempt < max_retries:
                    wait_time = base_delay * (2 ** attempt)  # Exponential backoff: 15, 30, 60, 120, 240
                    print(f"Query timed out. Retrying in {wait_time} seconds... (attempt {attempt + 1}/{max_retries})")
                    time.sleep(wait_time)
                    continue
            raise  # Re-raise if not a retryable error
    raise last_error

## Definitions

This is a critical area to visit with HMIS data standard updates and changes to values in fields

In [ ]:
# Cutoff dates for event filtering; includes events up to the current day
report_start_date = pd.to_datetime("1/1/2017").normalize()
report_end_date = pd.to_datetime("today").normalize()

# Project Type Code mappings
# 0: Emergency Shelter – Entry Exit
# 1: Emergency Shelter – Night-by-Night
# 2: Transitional Housing
# 3: PH – Permanent Supportive Housing (disability required for entry)
# 4: Street Outreach
# 5: RETIRED (HPRP)
# 6: Services Only
# 7: Other
# 8: Safe Haven
# 9: PH – Housing Only
# 10: PH – Housing with Services (no disability required for entry)
# 11: Day Shelter
# 12: Homelessness Prevention
# 13: PH – Rapid Re-Housing
# 14: Coordinated Entry

#Classifications for program types (using numeric codes)
homeless_program_types = [0, 1, 2, 4, 8]  # ES-EE, ES-NBN, TH, SO, Safe Haven

housing_program_types = [3, 9, 10, 13]  # PSH, PH-Housing Only, PH-Housing w/Services, RRH

housing_program_types_plus_prevention = housing_program_types + [12]  # Add Homelessness Prevention

homeless_program_types_no_outreach = [0, 1, 2, 8]  # ES-EE, ES-NBN, TH, Safe Haven
                                        
sheltered_homeless_program_types = [0, 1, 8]  # ES-EE, ES-NBN, Safe Haven
sheltered_ee_homeless_program_types = [0, 2, 8]  # ES-EE, TH, Safe Haven (entry-exit programs for ongoing enrollment events)

homeless_program_types_string = ",".join(str(x) for x in homeless_program_types)
not_homeless_program_types_string = ",".join(f"-{x}" for x in homeless_program_types)
housing_program_types_string = ",".join(str(x) for x in housing_program_types)

#These are situations from Exit Destinations, Prior Living Situations, and Current Living Situations

homelesssituation = ["Place not meant for habitation (e.g., a vehicle, an abandoned building, bus/train/subway station/airport or anywhere outside)",
                     "Safe Haven",
                     "Emergency shelter, including hotel or motel paid for with emergency shelter voucher, Host Home shelter"]


temporarysituation = ['Staying or living with family, temporary tenure (e.g., room, apartment, or house)', #Only in Exits
                     'Staying or living with friends, temporary tenure (e.g., room, apartment, or house)', #Only in Exits
                     'Residential project or halfway house with no homeless criteria', 
                    #  'Hotel or motel paid for without emergency shelter voucher',
                     'Host Home (non-crisis)', 
                     'Moved from one HOPWA funded project to HOPWA TH',
                     'Transitional housing for homeless persons (including homeless youth)']
                    #Visit temporary situation with community partners to determine if they are experiencing homelessness in our local definition, namely access to CE resoureces

permanentsituation = ["Moved from one HOPWA funded project to HOPWA PH",
                           "Rental by client, with ongoing housing subsidy",
                           "Rental by client, no ongoing housing subsidy",
                           "Owned by client, with ongoing housing subsidy",
                           "Owned by client, no ongoing housing subsidy",
                           "Staying or living with family, permanent tenure", #Only in Exits
                           "Staying or living with friends, permanent tenure"] #Only in Exits


# Definitions for shelter status

sheltered = ['Emergency shelter, including hotel or motel paid for with emergency shelter voucher, Host Home shelter', 
             'Safe Haven']
unsheltered = ['Place not meant for habitation (e.g., a vehicle, an abandoned building, bus/train/subway station/airport or anywhere outside)']
unknown_residences = ['Client prefers not to answer', 
                      "Client doesn't know", 
                      'Data not collected']


#Services keywords for pulling from Looker. Must specify case in these instances

housed_services = ["rent"
                   "Rent",
                   "Eviction Prevention",
                   "Eviction prevention",
                   "eviction prevention",
                   "Deposit",
                   "deposit",
                   "Security",
                   "security",
                   "Rental Assistance",
                   "rental assistance",
                   "Rental assistance"]

housed_services_string = ",".join(f"%{x}%" for x in housed_services)

## Extract Tables from Looker API for Transformation

### Enrollments

In [ ]:
body = {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "client_model", # Pulling from the client model, so we need to make sure we filter out deleted records from the client model we reference
    "fields": [
        "enrollments.id", #Enrollment ID
        "client_model.personal_id", #Personal ID
        "client_model.unique_identifier", #Client Unique ID
        "enrollments.start_date", #Enrollment Start Date
        "entry_screen.prior_residence_text", #Prior Residence
        "household_move_in_date.move_in_date", #Move In Date (from the household head)
        "programs.project_type_code", #Project Type
        "programs.raw_project_type_code", #Raw Project Type
        "enrollments.end_date", #Enrollment End Date
        "last_screen.exit_destination_text", #Exit Destination
        "entry_screen.health_dv_fleeing" #Fleeing DV
    ],
    "filters": {
        "enrollments.deleted": "No",
        "last_screen.deleted": "No",
        "entry_screen.deleted": "No",
        "programs.deleted": "No",
        "client_model.deleted": "No",
        "enrollments.start_date": "before today, today",
        "enrollments.end_date": "after 2017-01-01, NULL"
},
    "limit": -1 # This makes sure all records come in, not just the first 500 which is the default
}

# Run the inline query with retry logic
result = run_query_with_retry(sdk, body, result_format="csv")

# Convert to DataFrame and append to inventory
enrollments0 = pd.read_csv(StringIO(result))

In [ ]:
#Rename fields and type

columnHeaders={ "Enrollments Enrollment ID" : 'EnrollmentID',
               "Clients Personal ID": "PersonalID",
               "Clients Unique Identifier": "ClientUniqueIdentifier",
                "Enrollments Project Start Date": "ProjectStartDate",
                "Entry Screen Residence Prior to Project Entry" :"PriorResidence",
                "Enrollments Household Move-In Date": "HouseholdMoveInDate",
                "Programs Project Type Code": "ProjectTypeText",
                "Programs Project Type Code Raw": "ProjectTypeCode",
                "Enrollments Project Exit Date": "ProjectExitDate",
                "Update/Exit Screen Destination": "ExitDestination",
                "Entry Screen Currently Fleeing Domestic Violence": "CurrentlyFleeing"}

enrollments0.rename(columns=columnHeaders,inplace=True)

enrollments0["HouseholdMoveInDate"] = pd.to_datetime(enrollments0["HouseholdMoveInDate"], errors="coerce")
enrollments0["ProjectExitDate"] = pd.to_datetime(enrollments0["ProjectExitDate"], errors="coerce")
enrollments0["ProjectStartDate"] = pd.to_datetime(enrollments0["ProjectStartDate"], errors="coerce")

### Current Living Situation

In [ ]:
body = {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "base", # Pulling from the HMIS Performance model, so set the enrollments.date_filter to null to bring in all records (default is last quarter)
    "fields": [
        "enrollments.id", #Enrollment ID
        "clients.personal_id", #Personal ID
        "clients.unique_identifier", #Client Unique ID
        "current_living_situation.current_living_situation", #Current Living Situation
        "current_living_situation.information_date_date", #Current Living Situation Information Date
        "current_living_situation.id" #Current Living Situation ID
    ],
    "filters": {
        "enrollments.date_filter": "NULL", # Pull all records, not just the last quarter
        "current_living_situation.information_date_date" : "after 2017-01-01"}, #limit to only those records that have a current living situation after 2017-01-01. Change date as desired
    "limit": -1 # This makes sure all records come in, not just the first 500 which is the default
}

# Run the inline query with retry logic
result = run_query_with_retry(sdk, body, result_format="csv")

cls0 = pd.read_csv(StringIO(result))

In [ ]:
#Rename fields and type

columnHeaders={ "Enrollments Enrollment ID" : 'EnrollmentID',
               "Clients Personal ID": "PersonalID",
               "Clients Unique Identifier": "ClientUniqueIdentifier",
                'Current Living Situation Current Living Situation' : 'CurrentLivingSituation',
                'Current Living Situation Information Date' : 'InformationDate',
                "Current Living Situation ID": "CurrentLivingSitID",
                }

cls0.rename(columns=columnHeaders,inplace=True)

cls0["InformationDate"] = pd.to_datetime(cls0["InformationDate"], errors="coerce")


### Services Tables

Bring in services data in chunks from Looker based on these service types: 

1. Any non-bed night service in homeless program enrollments. These services indicate continued activity
2. Bed nights in a night-by-night shelter
3. Any housed service outside of a homeless program enrollment
4. Any outreach service in RRH program prior to move-in date

In Looker, services are tracked as an service *attendance* date or a service date. Bed-nights are considered an attendance date. These are tracked every night someone stays in a bed night. There are some other services (mostly for day centers) that are tracked at the attendance level. Other services are tracked as a plain service date. These queries are structured to capture the correct service item name, category, and date. There is opportunity for futher analysis if we want to:
- Bring in any other non housing bed-night service attendance dates for homeless programs to indicate continued activity beyond what is tracked in plain service dates 
- Consider other keywords for homeless services. Currently, just looking for outreach in RRH programs. Could revisit service attendance activity too. 
- Other keywords for housed services. These are defined in definitions portion of script. Could revisit service attendance activity too. 


In [ ]:
# 1. Any non-bed night service in homeless program enrollments. These services indicate continued activity

body = {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "base", # Pulling from the HMIS Performance model, so set the enrollments.date_filter to null to bring in all records (default is last quarter)
    "fields":[
            "enrollments.id", #Enrollment ID
            "clients.personal_id", #Personal ID
            "clients.unique_identifier", #Client Unique ID
            "programs.project_type_code", #Project Type
            "programs.raw_project_type_code", #Raw Project Type
            "service_items.id", #Service Item ID     
            "services.ref_category", #Service Category
            "service_items.service_item_name",  #Service Item Name
            "services.start_date_date"  #Service Start Date
        ],
        "filters":{
            "enrollments.date_filter": "NULL", # Pull all records, not just the last quarter
            "programs.raw_project_type_code" : homeless_program_types_string, #Only pull in homeless program types for this query
            "services.ref_category" : "-Housing", #, This filters out bed night services, which we pull in a different query
            "services.start_date_date" : "after 2017-01-01", #limit to only those records that have a services on or after 2017-01-01. Change date as desired
        },
        "limit":-1 # This makes sure all records come in, not just the first 500 which is the default
}

# Run the inline query with retry logic
result = run_query_with_retry(sdk, body, result_format="csv")

services1 = pd.read_csv(StringIO(result))

In [ ]:
# 2. Bed nights in a night-by-night shelter

body = {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "base", # Pulling from the HMIS Performance model, so set the enrollments.date_filter to null to bring in all records (default is last quarter)
    "fields":[
            "enrollments.id", #Enrollment ID
            "clients.personal_id", #Personal ID
            "clients.unique_identifier", #Client Unique ID
            "programs.project_type_code", #Project Type
            "programs.raw_project_type_code", #Raw Project Type
            "services.ref_category", #Service Category
            "service_dates.id", #Service Date ID
            "service_dates.date_date" #Service Date   
        ],
        "filters":{
            "enrollments.date_filter": "NULL", # Pull all records, not just the last quarter
            "programs.raw_project_type_code" : "1", #We only want to pull in bed nights from ES nbn for this query 
            "services.ref_category" : "Housing", #This filters for bed night services
            "service_dates.date_date" : "after 2017-01-01" #limit to only those records that have services on or after 2017-01-01. Change date as desired
        },
        "limit":-1 # This makes sure all records come in, not just the first 500 which is the default
}

# Run the inline query with retry logic
result = run_query_with_retry(sdk, body, result_format="csv")

services2 = pd.read_csv(StringIO(result))

In [ ]:
# 3. Any housed service outside of a homeless program enrollment. 

body = {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "base", # Pulling from the HMIS Performance model, so set the enrollments.date_filter to null to bring in all records (default is last quarter)
    "fields":[
            "enrollments.id", #Enrollment ID
            "clients.personal_id", #Personal ID
            "clients.unique_identifier",  #Client Unique ID
            "programs.project_type_code", #Project Type
            "programs.raw_project_type_code", #Raw Project Type
            "service_items.id", #Service Item ID
            "services.ref_category", #Service Category
            "service_items.service_item_name", #Service Item Name
            "services.start_date_date" #Service Start Date
        ],
        "filters":{
            "enrollments.date_filter": "NULL", # Pull all records, not just the last quarter
            "service_items.service_item_name" : housed_services_string,  #filter out for housed services as defined above
            "services.start_date_date" : "after 2017-01-01",  #limit to only those records that have a services on or after 2017-01-01. Change date as desired 
            "programs.project_type_code" : not_homeless_program_types_string # add a filter to limit to non-homeless program types. This would be redundant with the the first query
        },
        "limit":-1 # This makes sure all records come in, not just the first 500 which is the default
}

# Run the inline query with retry logic
result = run_query_with_retry(sdk, body, result_format="csv")

services3 = pd.read_csv(StringIO(result))

In [ ]:
# 4. Outreach services in housing programs prior to move-in date

body = {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "base", # Pulling from the HMIS Performance model, so set the enrollments.date_filter to null to bring in all records (default is last quarter)
    "fields":[
            "enrollments.id", #Enrollment ID
            "clients.personal_id",  #Personal ID
            "clients.unique_identifier", #Client Unique ID
            "programs.project_type_code", #Project Type
            "programs.raw_project_type_code", #Raw Project Type
            "service_items.id", #Service Item ID
            "services.ref_category", #Service Category
            "service_items.service_item_name", #Service Item Name
            "services.start_date_date",
            "household_move_in_date.move_in_date" #Move In Date (from the household head)
  
        ],
        "filters":{
            "enrollments.date_filter": "NULL", # Pull all records, not just the last quarter
            "service_items.service_item_name" : "%outreach%, %Outreach%",  # contains outreach
            "services.start_date_date" : "after 2017-01-01", #limit to only those records that have a services on or after 2017-01-01. Change date as desired 
            "programs.raw_project_type_code" : housing_program_types_string # add a filter to limit to housing projects
        },
        "limit":-1 # This makes sure all records come in, not just the first 500 which is the default
}

# Run the inline query with retry logic
result = run_query_with_retry(sdk, body, result_format="csv")

services4_0 = pd.read_csv(StringIO(result))

In [ ]:
services4 = services4_0[
    (services4_0['Enrollments Household Move-In Date'].isna()) |
    (services4_0['Enrollments Household Move-In Date'] >= services4_0['Services Start Date Date'])
].drop(columns=['Enrollments Household Move-In Date'])

In [ ]:
# Union all 4 service tables together
services = pd.concat([services1, services2, services3, services4], ignore_index=True)

In [ ]:
# Name and type the columns

columnHeaders={ "Enrollments Enrollment ID" : 'EnrollmentID', 
               "Clients Personal ID": "PersonalID",
               "Clients Unique Identifier": "ClientUniqueIdentifier",
                "Programs Project Type Code": "ProjectTypeText",
                "Programs Project Type Code Raw": "ProjectTypeCode",
                'Services Service Item ID' : 'ServicesID1', # this is for non-bed night services
                "Services Service Category" : 'ServiceCategory',
                'Services Service Item Name' : 'Text',
                'Services Start Date Date' : 'DateProvided1', # this is for non-bed night services
                'Service Attendance Dates ID' : 'ServicesID2', #this is for bed nights
                'Service Attendance Dates Service Attendance Date' : 'DateProvided2', #this is for bed nights
                }

services.rename(columns=columnHeaders,inplace=True)

services["DateProvided1"] = pd.to_datetime(services["DateProvided1"], errors="coerce")
services["DateProvided2"] = pd.to_datetime(services["DateProvided2"], errors="coerce")

In [ ]:
# Collapse the service attendance ID and date fields and service ID and date fields into one field

services['ServicesID'] = services['ServicesID1'].combine_first(services['ServicesID2'])
services['DateProvided'] = services['DateProvided1'].combine_first(services['DateProvided2'])

services.drop(columns=['ServicesID1', 'ServicesID2'], inplace=True)
services.drop(columns=['DateProvided1', 'DateProvided2'], inplace=True)

services.loc[
    (services['ServiceCategory'] == 'Housing') & (services['Text'].isna()),
    'Text'
] = 'Bed night'


In [ ]:
# Type all services as a homeless or housed service
# Opportunity to define keywords for services in this section

service_list = services[['ServiceCategory','Text']].drop_duplicates()

service_list['ServiceType'] = np.select(
    [
        service_list["Text"].str.contains("outreach", case=False, na=False) |  # Homeless services
        (service_list["Text"] == "Bed night"), # Homeless services
        service_list["Text"].str.contains("rental assistance", case=False, na=False) | #Housed services
        service_list["Text"].str.contains("eviction prevention", case=False, na=False) | #Housed services
        service_list["Text"].str.contains("security deposit", case=False, na=False) #Housed services
    ],
    ["Homeless", "Housed"],
    default=""
)



## Create Events Tables

This section transforms the raw HMIS data into homeless and housed (and deceased) events. 
- Current Living Situation Events
- Homeless Enrollments (homeless programs, currently fleeing, of homeless prior living siutation)
- Move-In Dates (for housing programs)
- Homeless Prevention Enrollments
- Service Events 
- Other Enrollment Events (exits and open enrollments)
- Deceased Events

### CLS Events

This code processes `cls0` containing Current Living Situation raw HMIS assessment data and generates a structured event table (`cls_events`). It filters relevant living situations, assigns event metadata, and classifies each event into a client status, event type, and shelter status based on predefined lists.

In [ ]:
cls_events = (
    cls0
    # Create initial EventTypeDetail column from raw CurrentLivingSituation
    .assign(EventTypeDetail=lambda d: d["CurrentLivingSituation"])
    # Filter rows to only those with recognized living situations
    .loc[lambda d: d["CurrentLivingSituation"].isin(homelesssituation + permanentsituation + temporarysituation)]
    # Add derived fields based on the living situation
    .assign(
        # Classify client's housing status based on CurrentLivingSituation
        ClientStatus=lambda d: np.select(
            [
                d["CurrentLivingSituation"].isin(homelesssituation),
                d["CurrentLivingSituation"].isin(permanentsituation),
                d["CurrentLivingSituation"].isin(temporarysituation)
            ],
            ['Homeless', # homeless situations
             'Housed', # permanent situations
             'Homeless'], # temporary situations 
            default=pd.NA
        ),
        # Define event type label based on CurrentLivingSituation
        EventType=lambda d: np.select(
            [
                d["CurrentLivingSituation"].isin(homelesssituation),
                d["CurrentLivingSituation"].isin(permanentsituation),
                d["CurrentLivingSituation"].isin(temporarysituation)
            ],
            ['Literally Homeless CLS', #homeless situations
             'Housed CLS', # permanent situations 
             'Unstable CLS'], # temporary situations
            default=pd.NA
        ),
        # Classify the sheltering condition of the client
        ClientShelterStatus=lambda d: np.select(
            [
                d["CurrentLivingSituation"].isin(sheltered),
                d["CurrentLivingSituation"].isin(unsheltered),
                d["CurrentLivingSituation"].isin(temporarysituation)
            ],
            ['Sheltered', #sheltered situations list
             'Unsheltered', # unsheltered situations list
             'Temporarily Housed'], # temporary situations list
            default=pd.NA
        ),
        EventDate=lambda d: pd.to_datetime(d["InformationDate"]),   # Use InformationDate as the timestamp for the event
        EventRecordType="Current Living Situation" # Mark record type explicitly
    )
        # Select final columns for event output
    .loc[:, [
        'PersonalID', 'ClientUniqueIdentifier', 'EventDate', 'ClientStatus', 'EventType', 'ClientShelterStatus', "EventRecordType", "EventTypeDetail", "EnrollmentID"
    ]]
)


### Homeless Enrollments Events

This code processes `enrollments0` containing raw HMIS enrollment data and generates a structured event table (`homeless_enrollment_events`). It filters relevant enrollments, assigns event metadata, and classifies each event into a client status, event type, and shelter status based on predefined lists.

> This table includes all homeless events

In [ ]:
homeless_enrollment_events = (
    enrollments0
    # Preserve original PriorResidence as EventTypeDetail for later reference
    .assign(EventTypeDetail=lambda d: d["PriorResidence"])
    # Filter for enrollments that meet homelessness-related criteria:
    .loc[lambda d: (
        d['PriorResidence'].isin(homelesssituation) | # - PriorResidence is literally homeless
        d['PriorResidence'].isin(temporarysituation) |     # - or temporarily housed
        (d['CurrentlyFleeing'] == "Yes") | # - or currently fleeing DV
        (d['ProjectTypeCode'].isin(homeless_program_types)) # - or enrolled in a known homeless program type
    )]
    # Create event-level fields to describe the enrollment
    .assign(
        ClientStatus="Homeless", # All in this table are considered homeless
        EventType="Homeless Enrollment", # Label for this event stream
        EventDate=lambda d: d["ProjectStartDate"], # Use enrollment start date as the event timestamp
        EventRecordType="Homeless Enrollment", # Label for this event type
        # Provide a more detailed event type based on the PriorResidence and CurrentlyFleeing
        EventTypeDetail=lambda d: np.select(
            [
                d['PriorResidence'].isin(homelesssituation), # - Use PriorResidence for known categories
                d['PriorResidence'].isin(temporarysituation),# - Use PriorResidence for known categories
                d['CurrentlyFleeing'] == "Yes" # - Label fleeing clients explicitly
            ],
            [d["PriorResidence"], # for homeless situations
             d["PriorResidence"], #for temporary situations
             "Fleeing DV"], # for fleeing clients
            default=d["ProjectTypeText"] # - Default to ProjectTypeText if unmatched
        ),
        # Classify ClientShelterStatus based on PriorResidence or program type
        ClientShelterStatus=lambda d: np.select(
            [
                d['PriorResidence'].isin(sheltered), # Matches known sheltered situations
                d['PriorResidence'].isin(unsheltered), # Matches known unsheltered situations
                d['ProjectTypeCode'] == 2, # Transitional Housing = Temporarily Housed
                d['ProjectTypeCode'].isin(sheltered_homeless_program_types), # ES-EE, ES-NBN, Safe Haven = Sheltered
                d['ProjectTypeCode'] == 4, # Street outreach is unsheltered
                d['PriorResidence'].isin(temporarysituation), # Temporary situations treated as temporarily housed
                (d['PriorResidence'].isin(unknown_residences) | d['PriorResidence'].isna()) # Unknown or missing values
            ],
            [
                "Sheltered",           # Matches known sheltered situations
                "Unsheltered",         # Matches known unsheltered situations
                "Temporarily Housed",  # Transitional Housing
                "Sheltered",           # ES-EE, ES-NBN, Safe Haven
                "Unsheltered",         # Street outreach is unsheltered
                "Temporarily Housed",  # Temporary situations treated as temporarily housed
                "Unknown"              # Unknown or missing values
            ],
            default="Sheltered"       # All others default to sheltered
        ),
        EnrollmentID=lambda d: d["EnrollmentID"] # Preserve Enrollment ID for linking
    )
    # Select final columns for output event table
    .loc[:, [
        'PersonalID', 'ClientUniqueIdentifier', 'EventDate', 'ClientStatus',
        'EventType', 'EventRecordType', 'EventTypeDetail',
        'ClientShelterStatus', 'EnrollmentID'
    ]]
)

### Move-In Date Events

This code processes `enrollments0` containing raw HMIS enrollment data and generates a structured event table (`move_in_date_events`). It filters relevant move-in dates, assigns event metadata, and classifies each event into a client status, event type, and shelter status based on predefined lists. 

> This table includes all housed events

In [ ]:
move_in_date_events = (enrollments0
                        # Initialize MoveInDate column from raw HouseholdMoveInDate
                       .assign(MoveInDate = lambda d: pd.to_datetime(d['HouseholdMoveInDate'], errors='coerce'))
                        # Filter to rows where:
                       .loc[lambda d: (d['MoveInDate'] <= report_end_date) & # - MoveInDate is on or before the report_end_date
                                   (d['ProjectTypeCode'].isin(housing_program_types))]  # - ProjectTypeCode is a known permanent housing type
                       .assign(ClientStatus = "Housed", # Mark client as housed upon move-in
                               EventType = "Housing Move-In Date", # Label the type of system event
                               EventDate = lambda d: d["MoveInDate"], # Use actual move-in date as event timestamp
                               EventRecordType = "Move-In", # Record type label for categorization
                               EventTypeDetail = lambda d: d["ProjectTypeText"],  # Track what type of housing program
                               EnrollmentID = lambda d: d["EnrollmentID"]) # Preserve Enrollment ID for joins
                        # Select only final output fields used in the systemwide event table
                       .loc[:, ['PersonalID', 'ClientUniqueIdentifier', 'EventDate', 'ClientStatus', 'EventType', "EventRecordType", "EventTypeDetail", "EnrollmentID"]]
)

### Homelessness Prevention Events

This code creates a structured event table `prevention_enrollment_events` for enrollments into Homelessness Prevention projects. Homelessness Prevention projects aim to help clients avoid entering homelessness, so all clients enrolled are considered housed.

Key logic:
- Filters to only include enrollments where ProjectTypeCode is "Homelessness Prevention"
- Assigns a "Housed" ClientStatus and labels the event as "Prevention Enrollment"
- Uses the enrollment start date as the event date
- Outputs standardized event fields for systemwide integration and 

> This table includes all housed events

In [ ]:

prevention_enrollment_events = (
    enrollments0
    # Filter for enrollments that are part of Homelessness Prevention programs
    .loc[lambda d: d["ProjectTypeCode"] == 12]
    # Assign event fields that define the prevention enrollment record
    .assign(
        ClientStatus="Housed", # Prevention programs serve clients who are housed
        EventType="Prevention Enrollment", # Label the event type for this table
        EventDate=lambda d: d["ProjectStartDate"], # Use enrollment start date as the event timestamp
        EventRecordType="Prevention Enrollment", # Label the event record type for this table
        EventTypeDetail=lambda d: d["ProjectTypeText"], # Keep the original project type for detailed breakdowns
        EnrollmentID=lambda d: d["EnrollmentID"] # Preserve Enrollment ID for linkage and tracking
    )
    # Select final set of fields to include in the output event table
    .loc[:, [
        'PersonalID', 'ClientUniqueIdentifier', 'EventDate', 'ClientStatus', 'EventType', "EventRecordType", "EventTypeDetail", "EnrollmentID"
    ]]
)

### Service Events

This block processes the raw `services` data to generate a unified event table for `service_events`. It merges in classifications (service_list), filters to relevant records, and assigns key event fields such as EventType, ClientStatus, and ClientShelterStatus. 

In [ ]:
service_events = (
    services
    # Merge in service type classifications from service_list to enrich service records
    .merge(
        service_list,
        on=["ServiceCategory", "Text"], # Match on service category and name
        how="left" # Keep all services, even if classification is neither housed nor homeless service event
    )
    # Filter
    .loc[lambda d: (d["ServiceType"] != "") | # - Keep services with a non-empty classification (ServiceType) that is Housed or Homeless service, OR
                   (d['ProjectTypeCode'].isin(homeless_program_types))] # - Any service provided by a homeless program (even if unclassified) to show continued activity in the program
    # Assign event attributes
    .assign(
        # Classify event type based on service name or classification
        EventType=lambda d: np.select(
            # IF
            [
                d["Text"] == "Bed night", # Emergency shelter stays
                d["ServiceType"] == "Housed", # Prevention/housing assistance
                d["ServiceType"] == "Homeless" # Outreach or crisis engagement
            ],
            #THEN 
            ["Shelter Bed Night",  # Emergency shelter stays
             "Rent or Deposit Service", # Prevention/housing assistance
             "Outreach Contact Service"], # Outreach or crisis engagement
            default="Service From Homeless-Only Program" # Catch-all for uncategorized services from homeless programs
        ),
        # Define client status based on service type
        ClientStatus=lambda d: np.where(d["ServiceType"] == "", "Homeless", d["ServiceType"]),
         # Determine shelter status based on project type
        ClientShelterStatus=lambda d: np.select(
            #IF
            [
                d['ClientStatus'] == "Housed",
                d['ProjectTypeCode'] == 2, # Transitional Housing
                d['ProjectTypeCode'].isin(sheltered_homeless_program_types), # Emergency Shelter or Safe Haven
                d['ProjectTypeCode'] == 4, # Street Outreach
                (d['ProjectTypeCode'] == 13) & (d['ClientStatus'] == "Homeless") # PH – Rapid Re-Housing
            ],
            #THEN
            [pd.NA, # Housed clients do not have a shelter status
             "Temporarily Housed", #TH service
             "Sheltered",  # ES or safe haven 
             "Unsheltered", # Street-based outreach
             "Unknown" # outreach services in RRH programs prior to move-in 
             ],
            default=pd.NA # Leave blank if indeterminate
        ),
        EventDate=lambda d: pd.to_datetime(d["DateProvided"]),  # Use service delivery date as the event timestamp
        EventRecordType="Service",  # Label all records as service-type events
        EventTypeDetail=lambda d: d["Text"],  # Preserve original service text as event detail
    )
     # Select and order final fields for event table output
    .loc[:, [
        'PersonalID', 'ClientUniqueIdentifier', 'EventDate', 'ClientStatus', 'ClientShelterStatus', 'EventType', 'EventRecordType', 'EventTypeDetail', "EnrollmentID"
    ]]
)


### Other Enrollment Events

This code constructs events from `enrollments0` that are not captured by move-in dates, CLS records, or prevention logic—particularly open enrollments and exits from housing and homelessness programs. It determines client status, event type, and shelter status based on ExitDestination, ProjectTypeCode, and HouseholdMoveInDate. The logic includes special handling for open enrollments in transitional housing, homeless prevention, and homeless programs without outreach.

In [ ]:
other_enrollment_events = (
    enrollments0
    .assign(
        HouseholdMoveInDate=pd.to_datetime(enrollments0["HouseholdMoveInDate"], errors='coerce') # Initialize MoveInDate column from raw HouseholdMoveInDate
    )
    # Filter rows that meet one of two conditions:
    .loc[lambda d: (
        (d["ProjectExitDate"].isna() & d["ProjectTypeCode"].isin(homeless_program_types_no_outreach + housing_program_types_plus_prevention)) | # 1. Still enrolled (no exit date) AND project is a homeless or housing program OR
        d["ExitDestination"].isin(homelesssituation + permanentsituation + temporarysituation)  | # 2. Exited to a destination that indicates homelessness status (homeless, permanent, or temporary)
        (d["ExitDestination"].isin(unknown_residences) & d["ProjectTypeCode"].isin(homeless_program_types_no_outreach)) # 3. Exited to an unknown or missing destination from a homeless program
    )]
    # Assign key event fields
    .assign(
        # Set EventDate:
        # - Use exit date if available
        # - Otherwise use the end of the reporting period for open enrollments
        EventDate=lambda d: d["ProjectExitDate"].fillna(report_end_date),
         # Determine ClientStatus:
            # Housed if:
            # - exited to a permanent housing destination, OR
            # - still enrolled in a housing or prevention project AND
            #   has a valid move-in date on or before the report end date
        ClientStatus = lambda d: np.select(
            [
                (
                    d["ExitDestination"].isin(permanentsituation) |
                    (
                        d["ProjectTypeCode"].isin(housing_program_types) &
                        d["ProjectExitDate"].isna() &
                        (d["HouseholdMoveInDate"] <= report_end_date)
                    ) |
                    (
                        (d["ProjectTypeCode"] == 12) &
                        d["ProjectExitDate"].isna()
                    )
                )
            ],
            ["Housed"],
             # All other cases default to "Homeless"
             # - applies to exits to homeless or temporary destinations
             # - or open enrollments without move-in
            default="Homeless" 
        ),
        # Determine EventType based on exit destination
        EventType=lambda d: np.select(
            [
                d["ExitDestination"].isin(homelesssituation),
                d["ExitDestination"].isin(permanentsituation),
                d["ExitDestination"].isin(temporarysituation),
                d["ExitDestination"].isin(unknown_residences)
            ],
            ["Homeless Exit From Program", # Exit to a homeless destination 
             "Housed Exit From Program",  # Exit to a permanent housing destination
             "Unstable Exit From Program", # Exit to a temporary situation
             "Unknown Exit From Homeless Program"], # Exit to an unknown or missing destination from a homeless program
            default="Still Enrolled In Program" # Default to still enrolled if no exit destination
        ),
        # Label record based on whether it's an exit or still-open enrollment
        EventRecordType=lambda d: np.where(
            pd.notnull(d["ExitDestination"]), # If there is an exit destination
            "Exit Destination",  # Label as exit destination if applicable
            "Open Enrollment" # Label as open enrollment if no exit destination
        ),
        # For detail, use ExitDestination if informative, otherwise fallback to ProjectTypeCode
        EventTypeDetail=lambda d: np.where(
            d["ExitDestination"].isin(homelesssituation + permanentsituation + temporarysituation),
            d["ExitDestination"], # Use ExitDestination for known categories
            d["ProjectTypeText"] # Default to ProjectTypeText if no exit destination
        ),
        # Determine shelter status based on project type, status, and prior residence
        ClientShelterStatus=lambda d: np.select(
            [
                d["ClientStatus"].isin(["Housed"]), # No shelter status for housed clients
                (d["ProjectTypeCode"] == 2) & (d["EventType"] != "Unknown Exit From Homeless Program"), # Transitional = temporarily housed
                (d["EventType"] == "Still Enrolled In Program") & d["ProjectTypeCode"].isin(homeless_program_types_no_outreach), # Still enrolled in a homeless program with no outreach = sheltered
                (d["EventType"] == "Homeless Exit From Program") & d["ExitDestination"].isin(sheltered), # Exit to a homeless destination (sheltered)
                (d["EventType"] == "Homeless Exit From Program") & d["ExitDestination"].isin(unsheltered),# Exit to a homeless destination (unsheltered)
                (d["EventType"] == "Unstable Exit From Program") & d["ExitDestination"].isin(temporarysituation), #Temporarily Housed
                (d["EventType"] == "Still Enrolled In Program") & # Still enrolled ...  
                    d["ProjectTypeCode"].isin(housing_program_types) & # still in PH without move-in date AND
                    d["PriorResidence"].isin(unsheltered), #PriorResidence is unsheltered
                (d["EventType"] == "Unknown Exit From Homeless Program") | # Unknown exit from homeless program OR
                    ((d["EventType"] == "Still Enrolled In Program") &  # Still enrolled in ... 
                        d["ProjectTypeCode"].isin(housing_program_types) & # still in PH without move-in date AND
                        (d["PriorResidence"].isin(unknown_residences) | d["PriorResidence"].isna())) #PriorResidence is unknown
            ],
            [
                pd.NA,                  # No shelter status for housed clients
                "Temporarily Housed",   # Transitional Housing
                "Sheltered",            # Open homeless program
                "Sheltered",            # Exit to sheltered location
                "Unsheltered",          # Exit to unsheltered locationd
                "Temporarily Housed",   # Exit to temporary situation
                "Unsheltered",          # PH open enrollment with unsheltered prior
                "Unknown"               # PH open enrollment with unknown prior
            ],
            default="Sheltered" # Default to sheltered if not matched
        ),
        EnrollmentID=lambda d: d["EnrollmentID"]  # Include EnrollmentID for linkage
    )
        .assign(EventDate=lambda d: pd.to_datetime(d['EventDate'].astype(str), errors='coerce')) # Ensure EventDate is stored as datetime (may have mixed types from np.where)
    # Final selected fields for the event output
    .loc[:, [
        'PersonalID', 'ClientUniqueIdentifier', 'EventDate', 'ClientStatus', 'EventType', 'ClientShelterStatus', 'EventRecordType', 'EventTypeDetail', 'EnrollmentID'
    ]]
)

### Ongoing Enrollment Events

This code constructs events from homelessness programs in `enrollments0` that span multiple months to ensure calculated episodes span entire enrollment periods. Events are added for any contained day at least 14 days from the start and end of a residential homeless enrollment that is the 1st or 15th of a month.

In [ ]:

df = (
    enrollments0
    .loc[lambda d: d["ProjectTypeCode"].isin(sheltered_ee_homeless_program_types)]
    .assign(
        ProjectStartDate=lambda d: pd.to_datetime(d["ProjectStartDate"], errors="coerce"),
        ProjectExitDate=lambda d: pd.to_datetime(d["ProjectExitDate"], errors="coerce"),
        calc_start=lambda d: d["ProjectStartDate"].clip(lower=report_start_date),
        calc_end=lambda d: d["ProjectExitDate"].fillna(report_end_date).clip(upper=report_end_date),
    )
)

calc_end = (
    df["ProjectExitDate"]
    .fillna(report_end_date)
    .clip(upper=report_end_date)
)

# ---- Filter to enrollments longer than 14 days ----
relevant = calc_end > df["calc_start"] + pd.Timedelta(days=14)
df = df.loc[relevant].reset_index(drop=True)
calc_end = calc_end.loc[relevant].reset_index(drop=True)

# ---- Month ordinals (YYYY*12 + MM) ----
start_ord = df["calc_start"].dt.year * 12 + (df["calc_start"].dt.month - 1)
end_ord   = calc_end.dt.year * 12 + (calc_end.dt.month - 1)

n_months = end_ord - start_ord

# ---- Expand rows by number of months ----
rep = np.repeat(np.arange(len(df)), n_months + 1)

# Month offsets within each enrollment
month_offset = np.concatenate([np.arange(m + 1) for m in n_months])

# Base month ordinal
base_ord = start_ord.values[rep] + month_offset

# base_ord is "months since 0" style
years  = (base_ord // 12).astype(int)
months = (base_ord % 12 + 1).astype(int)

base_month = pd.to_datetime(
    {"year": years, "month": months, "day": 1},
    errors="raise"
)

# ---- Candidate dates: 1st & 15th ----
event_dates = np.concatenate([
    base_month,
    base_month + pd.Timedelta(days=14)
])

rep2 = np.tile(rep, 2)

# ---- Build final frame ----
events = pd.DataFrame({
    "PersonalID": df.loc[rep2, "PersonalID"].values,
    "ClientUniqueIdentifier": df.loc[rep2, "ClientUniqueIdentifier"].values,
    "EnrollmentID": df.loc[rep2, "EnrollmentID"].values,
    "ProjectTypeCode": df.loc[rep2, "ProjectTypeCode"].values,
    "ProjectTypeText": df.loc[rep2, "ProjectTypeText"].values,
    "calc_start": df.loc[rep2, "calc_start"].values,
    "calc_end_date": calc_end.loc[rep2].values,
    "EventDate": event_dates
})

# ---- Filter strictly between start and end, including only gaps larger than 15 days ----
ongoing_enrollment_events = (
    events.loc[
        (events["EventDate"] >= (events["calc_start"] + pd.Timedelta(days=14))) &
        (events["EventDate"] <= (events["calc_end_date"] - pd.Timedelta(days=14)))
    ]
    # some people have overlapping (read: likely invalid) enrollments, so drop duplicates on Client+Date
    .drop_duplicates(subset=["ClientUniqueIdentifier", "EventDate"])
    .assign(
        ClientStatus = "Homeless", # only kept homeless projects
        EventType = "Ongoing Enrollment", # label the event type for this table
        EventRecordType ="Ongoing Enrollment", # Label the event record type for this table
        EventTypeDetail=lambda d: d["ProjectTypeText"], # Keep the original project type for detailed breakdowns
        # Set ClientShelterStatus based on project type: TH = Temporarily Housed, others = Sheltered
        ClientShelterStatus=lambda d: np.where(
            d["ProjectTypeCode"] == 2,  # Transitional Housing
            "Temporarily Housed",
            "Sheltered"  # ES-EE and Safe Haven
        )
    )
    # Select final set of fields to include in the output event table
    .loc[:, [
        'PersonalID', 'ClientUniqueIdentifier', 'EventDate', 'ClientStatus', 'EventType', "EventRecordType", 
        "ClientShelterStatus","EventTypeDetail", "EnrollmentID"
    ]]
)

In [ ]:
ongoing_enrollment_events

### Deceased Events

This code identifies enrollments in `enrollments0` where the client’s recorded exit destination is “Deceased”. It creates a standardized event record capturing the death as an exit from the system, assigning a status of "Deceased" and using the exit date or report end date as the event timestamp. 

> All events are recorded as deceased

In [ ]:
deceased_events = (
    enrollments0
    # copy ExitDestination to EventTypeDetail
    .assign(
        EventTypeDetail=lambda d: d["ExitDestination"]
    )
    # Filter for enrollments where the client exited due to death
    .loc[lambda d: d["ExitDestination"] == "Deceased"]
    # Assign key event metadata
    .assign(
        EventDate=lambda d: np.where(d["ProjectExitDate"].isna(), 
                                     report_end_date, 
                                     d["ProjectExitDate"]),  # Use today's date only if exit date is missing
        ClientStatus="Deceased", # Mark client as deceased
        EventType="Exit Due to Death",
        EventRecordType="Exit Destination",
        EnrollmentID=lambda d: d["EnrollmentID"]
    )
    .assign(
        EventDate=lambda d: pd.to_datetime(d["EventDate"])
    )
    # Select and organize final output fields
    .loc[:, [
        "PersonalID", "ClientUniqueIdentifier", "EventDate", "ClientStatus", "EventType", "EventRecordType", "EventTypeDetail", "EnrollmentID"
    ]]
)

## Compile Events Into all_events

This code concatenates all event tables into a single all_events DataFrame `all_events`. 

It ensures consistent datetime formatting, filters by report end date, and enforces de-duplication rules to retain only the most relevant event per person per date. If there is more than one event on the same day for a person, the code will prioritize in this order:
1. Deceased
2. Homeless
3. Housed
4. Unknown

It also includes logic to handle events after a client has marked as deceased. If there are only “still enrolled” events after a death, we assume the program hadn’t updated their records yet, and we keep the death. But if there are meaningful events after the death, we assume the death was an error and remove it.

In [ ]:
# ---- 1. Concatenate FIRST ----
all_events = pd.concat(
    [
        cls_events,
        homeless_enrollment_events,
        move_in_date_events,
        service_events,
        other_enrollment_events,
        deceased_events,
        prevention_enrollment_events,
        ongoing_enrollment_events
    ],
    ignore_index=True
)

# ---- 2. Convert EventDate ONCE ----
all_events["EventDate"] = pd.to_datetime(all_events["EventDate"], errors="coerce")

# ---- 3. Drop exact duplicates ----
all_events = all_events.drop_duplicates()

# ---- 4. Filter to report period early ----
all_events = all_events.loc[all_events["EventDate"] <= report_end_date]

# ---- 5. Vectorized death logic (NO apply) ----
is_death = all_events["EventType"] == "Exit Due to Death"

# Max death date per person
death_date = (
    all_events.loc[is_death]
    .groupby("PersonalID")["EventDate"]
    .max()
)

all_events = all_events.merge(
    death_date.rename("DeathDate"),
    on="PersonalID",
    how="left"
)

merged_is_death = all_events["EventType"] == "Exit Due to Death"
is_still = all_events["EventType"] == "Still Enrolled in Program"

# Is there a real event AFTER death?
has_event_after_death = (
    (all_events["EventDate"] > all_events["DeathDate"]) &
    (~is_still)
)

invalid_death = (
    has_event_after_death
    .groupby(all_events["PersonalID"])
    .transform("any")
    .fillna(False)
    .astype(bool)
)

all_events = all_events.loc[
    ~(
        (invalid_death & merged_is_death) |
        (~invalid_death & is_still & all_events["DeathDate"].notna())
    )
]

# ---- 6. Assign status priority ----
status_order = {
    "Deceased": 0,
    "Housed": 1,
    "Homeless": 2,
    "Unknown": 3
}

all_events["ClientStatusPriority"] = (
    all_events["ClientStatus"]
    .map(status_order)
    .fillna(4)
)

all_events = (
    all_events
    # ---- ONE final sort ----
    .sort_values(by=["PersonalID", "EventDate", "ClientStatusPriority"], ascending=[True, False, True])
    # ---- Deduplicate per person/day ----
    .drop_duplicates(subset=["PersonalID", "EventDate"], keep="first")
    .drop(columns=["ClientStatusPriority", "DeathDate"])
    .reset_index(drop=True)
)

# Create a unique EventID
This code generates a compact, consistent EventID for each row in the all_events DataFrame by hashing a combination of PersonalID, EventDate, and EnrollmentID. The result is a short, unique string that can be used to identify and reference specific events across systems or tables without exposing sensitive client details or relying on unstable natural keys.

Using the first 16 characters of an MD5 hash provides a high degree of uniqueness while keeping the identifier compact and readable. The full MD5 hash is 32 characters, but the first 16 are sufficient to avoid collisions in most practical datasets—especially when the input string includes three strongly identifying fields.

In [ ]:
import hashlib

def make_event_id(row):
    string_to_hash = f"{row['PersonalID']}_{row['EventDate']}_{row['EnrollmentID']}"
    return hashlib.md5(string_to_hash.encode()).hexdigest()[:16]

all_events['EventID'] = all_events.apply(make_event_id, axis=1)

### Data Typing, Ordering, and Naming

In [ ]:

all_events['EnrollmentID'] = all_events['EnrollmentID'].apply(lambda x: str(int(x)) if pd.notnull(x) else None) #make sure EnrollmentID is a string without trailing decimal and zeros

In [ ]:

all_events['EventDate'] = pd.to_datetime(all_events['EventDate'], errors='coerce')
all_events['EventDate'] = pd.to_datetime(all_events['EventDate'].dt.date)  # this preserves datetime64[ns] with 00:00:00 time

In [ ]:
# Define the target column order and types
column_types = {
    'EventID': 'string',
    'PersonalID': 'string',
    'ClientUniqueIdentifier': 'string',
    'EnrollmentID': 'string',
    'EventDate': 'datetime64[ns]',
    'ClientStatus': 'string',
    'ClientShelterStatus': 'string',
    'EventType': 'string',
    'EventRecordType': 'string',
    'EventTypeDetail': 'string'
}

# Select and cast columns in the desired order
final = all_events[list(column_types.keys())].copy()
final = final.astype(column_types)

### Create Data As of Date

In [ ]:
final['DataAsOfDate'] = pd.to_datetime(date.today()) 


This format is needed for the parquet to work

In [ ]:
final["EventDate"] = final["EventDate"].dt.strftime("%Y-%m-%d")
final["DataAsOfDate"] = final["DataAsOfDate"].dt.strftime("%Y-%m-%d")


## Load

In [ ]:
# #For development outside of azure

# import io
# from azure.storage.blob import BlobServiceClient
# from azure.identity import DefaultAzureCredential

# # Define a function to write a DataFrame to Azure Blob Storage as a Parquet file
# def write_table(df, file_name):
#     # Define the storage account and container name
#     storage_account_name = "[[AZURE_STORAGE_ACCOUNT_NAME]]"
#     container_name = "[[AZURE_CONTAINER_NAME]]"

#     # Authenticate using Microsoft Entra credentials
#     credential = DefaultAzureCredential()

#     # Create a BlobServiceClient
#     blob_service_client = BlobServiceClient(
#         account_url=f"https://{storage_account_name}.blob.core.windows.net",
#         credential=credential
#     )

#     # Create a BlobClient for the specified blob
#     blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_name)

#     # Convert DataFrame to Parquet format in memory
#     parquet_buffer = io.BytesIO()
#     df.to_parquet(parquet_buffer, engine="pyarrow", index=False)

#     # Upload the Parquet data to Azure Blob Storage
#     blob_client.upload_blob(parquet_buffer.getvalue(), overwrite=True)

# # Write the 'final' DataFrame to Blob Storage
# write_table(final, "[[OUTPUT_PARQUET_FILENAME]]")

In [ ]:
# #For Production inside  azure

# # Initiate Parameter Fields.
# container_name = "[[AZURE_CONTAINER_NAME]]"
# file_name = "[[OUTPUT_PARQUET_FILENAME]]"

# # Create a BlobServiceClient object using the connection string
# connection_string = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]', '[[ADLS_CONNECTION_SECRET]]')
# blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# # Create a BlobClient for the final blob
# final_blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_name)

# # Save DataFrame to a temporary Parquet file on Azure Blob Storage
# with BytesIO() as temp_buffer:
#     final.to_parquet(temp_buffer, engine='pyarrow', index=False)
#     final_blob_client.upload_blob(temp_buffer.getvalue(), overwrite=True)